In [13]:
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import random_split
from torch_geometric.data import DataLoader

from halide_gnn_cost_model.data import PipelineDataset
from halide_gnn_cost_model.data import graph_transformer_preprocessor
from halide_gnn_cost_model.model import PipeGPS

In [ ]:
PIPELINES_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/pipelines-16k-data")
GRAPH_TRANSFORMER_MODEL_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/graph_transformer")
GRAPH_TRANSFORMER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
USE_GPU = True

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: mps


In [4]:
# Load dataset
dataset = PipelineDataset(PIPELINES_DIR, preprocessor=graph_transformer_preprocessor)

1lines [00:00, 2470.14lines/s]
1lines [00:00, 33554.43lines/s]
1lines [00:00, 43240.25lines/s]
1lines [00:00, 55924.05lines/s]


In [10]:
# Train/test split
num_train = int(0.95 * len(dataset))
train_dataset, test_dataset = random_split(dataset, [num_train, len(dataset) - num_train])
len(train_dataset), len(test_dataset)

(15605, 822)

In [5]:
data = dataset[0].clone()
data

Data(y=[5], edge_index=[2, 66], x=[31, 1], type=[31, 1], node_type=[31], edge_type=[66], node_token=[31], laplacian_eigenvector_pe=[31, 8], random_walk_pe=[31, 8], degree_pe=[31, 8], pe=[31, 24], total_token_vocab=23)

# Model

In [12]:
vocab_size = len(dataset.ast_vocab) + len(dataset.sched_vocab) + 1
model = PipeGPS(
    hidden_channels=64,
    num_layers=8,
    num_attn_heads=4,
    attn_type="multihead",
    attn_kwargs={"dropout": 0.5},
    vocab_size=vocab_size,
    num_runtime_targets=5,
)
model = model.to(device)

In [ ]:
pe = torch.cat((data.laplacian_eigenvector_pe, data.random_walk_pe, data.degree_pe), dim=-1)
pe = pe.to(device)
x = data.type.to(device)
edge_index = data.edge_index.to(device)
node_type = data.node_type.to(device)
res = model(x, pe, node_type, edge_index)
res

tensor([[-4.6492,  1.4575, 10.1726, -0.4714,  4.9321]], device='mps:0',
       grad_fn=<LinearBackward0>)

# Train model

In [11]:
data_loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)
data_loader

/var/folders/qh/c7l883gn5s548l5w18l25cwc0000gn/T/nix-shell.s283x9/ipykernel_78078/958583031.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  data_loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)


In [14]:
for data in data_loader:
    break

In [20]:
x = data.type
edge_index = data.edge_index
pe = torch.cat((data.laplacian_eigenvector_pe, data.random_walk_pe, data.degree_pe), dim=-1)
node_type = data.node_type
res = model(x, pe, node_type, edge_index, data.batch)

In [ ]:
# Save model every n epochs
SAVE_EVERY = 5
NUM_EPOCHS = 5

# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.MSELoss()

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    total_loss = 0
    for batch_idx, data in enumerate(data_loader):
        optimizer.zero_grad()
        data = data.to(device)
        x = data.type
        edge_index = data.edge_index
        pe = torch.cat((data.laplacian_eigenvector_pe, data.random_walk_pe, data.degree_pe), dim=-1)
        node_type = data.node_type
        pred = model(x, pe, node_type, edge_index, data.batch)
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), torch.log(data.y))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    # Print average loss for the epoch
    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    # Save model
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(model.state_dict(), GRAPH_TRANSFORMER_MODEL_DIR / f"pipeline_model_epoch_{epoch+1}.pt")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1, Loss: 28.8609
